from plotting_celltypes_new import (get_animal_clean_dict_activity, fit_GLM_population, get_residual_activity_dict)
import pickle
import numpy as np
import matplotlib.pyplot as plt
import torch
import slicetca

In [18]:
from plotting_celltypes_new import (fit_GLM_population, get_residual_activity_dict, has_run_of_n_nans, interp_nans_1d)
import pickle
import numpy as np
import matplotlib.pyplot as plt
import torch
import slicetca
import h5py

In [33]:

def get_animal_clean_dict_activity(filepath, use_final=True):

    with h5py.File(filepath, 'r') as f:
        if use_final:
            animal_group = f['animals']
        else:
            animal_group = f['animal']

        print(f"animal_group.keys() {animal_group.keys()}")

        shiftR_refs = animal_group['ShiftR'][:]

        shiftRunning_refs = animal_group['ShiftRunning'][:]

        if use_final:
            shiftL_refs = animal_group['ShiftL'][:]
        else:
            shiftL_refs = animal_group['ShiftLrate'][:]

        animal_clean_dict_activity = {}


        animal_trials_original = []
        animal_trials_clean = []

        count = 0

        animal_vel_dict = {}
        animal_lick_dict = {}

        trials_to_remove_local = []
        for animal_idx in range(len(shiftR_refs)):
            delta_f = np.array(f[shiftR_refs[animal_idx][0]])

            animal_trials_original.append(delta_f.shape[1])


            # nan_trials = np.any(np.isnan(delta_f), axis=(0, 2))

            # delta_f_clean = delta_f[:, ~nan_trials, :]

            # delta_f_clean = delta_f

            vel = f[shiftRunning_refs[animal_idx][0]]
            vel = np.array(vel).T

            lick = f[shiftL_refs[animal_idx][0]]
            lick = np.array(lick).T

            # vel_nan_mask = np.isnan(vel)

            # plt.imshow(vel_nan_mask, aspect='auto')
            # plt.title(f"animal_idx {animal_idx} nans={np.any(vel_nan_mask)}")
            # plt.show()


            vel_clean = np.empty(vel.shape)
            lick_clean = np.empty(lick.shape)

        
            trials_to_remove_list = []

            delta_f_clean = np.empty(delta_f.shape)


            # print(f"delta_f.shape {delta_f.shape}")

            
            if animal_idx == 22:
                valid_cells = range(1, delta_f.shape[0])
            else:
                valid_cells = range(delta_f.shape[0])

            for cell in valid_cells: #range(delta_f.shape[0]):
                
            
                cell_data = delta_f[cell,:,:].T

                # nan_mask = np.isnan(cell_data)

                # cell_list_num_trials.append(np.sum(nan_mask, axis=0))


                for trial in range(cell_data.shape[1]):

                    trial_data = cell_data[:, trial]

                    vel_data_trial = vel[:,trial]
                    lick_data_trial = lick[:,trial]

                    if np.any(np.isnan(trial_data)) or np.any(np.isnan(vel_data_trial)):

                        has_5_nans = has_run_of_n_nans(trial_data, n=5)

                        has_5_nans_vel = has_run_of_n_nans(vel_data_trial, n=5)
                        
                        if has_5_nans or has_5_nans_vel:                            
                            if count == 105:
                                trials_to_remove_local.append(trial)

                            

                            # plt.plot(trial_data)
                            # plt.title(f"Before More than 5 nans cell={count} trial={trial}")
                            # plt.show()

                            # plt.plot(trial_data)
                            # plt.title(f"After More than 5 nans cell={count} trial={trial}")
                            # plt.show()

                            if trial not in trials_to_remove_list:
                                trials_to_remove_list.append(trial)
                        else:

                            # if count==30:

                            #     plt.plot(trial_data)
                            #     plt.title(f"Before Less than 5 nans cell={count} trial={trial}")
                            #     plt.show()

                            trial_data = interp_nans_1d(trial_data)
                            delta_f_clean[cell, trial, :] = trial_data

                            trial_data_vel = interp_nans_1d(vel_data_trial)
                            vel_clean[:,trial] = trial_data_vel

                            trial_data_lick = interp_nans_1d(lick_data_trial)
                            lick_clean[:,trial] = trial_data_lick





                            # if count==30:
                            #     plt.plot(trial_data)
                            #     plt.title(f"After Less than 5 nans cell={count} trial={trial}")
                            #     plt.show()

                    else:
                        delta_f_clean[cell, trial, :] = trial_data

                        vel_clean[:,trial] = vel_data_trial 
                        lick_clean[:,trial] = lick_data_trial 


                count+=1




            trials_to_remove_array = np.array(trials_to_remove_list) 

            if len(trials_to_remove_array) !=0:

                # print(f"trials_to_remove_array {trials_to_remove_array}")
                mask = np.ones(cell_data.shape[1], dtype=bool)
                mask[trials_to_remove_array] = False
                delta_f_clean = delta_f_clean[:, mask,:]

                vel_clean = vel_clean[:, mask]
                lick_clean = lick_clean[:, mask]

            animal_trials_clean.append(delta_f_clean.shape[1])

            cell_dict = {}


            for cell in valid_cells:#range(delta_f.shape[0]):

                cell_data = delta_f_clean[cell,:,:]
                
                # print(f"nans early on {np.any(np.isnan(cell_data))}")

                # cell_data = (cell_data - np.mean(cell_data)) / np.std(cell_data)

                # cell_dict[f"cell_{cell+1}"] = cell_data.T



                mean = np.mean(cell_data)
                std = np.std(cell_data)

                if std == 0 or not np.isfinite(std):
                    print(f" -> zero or bad std for this cell, skipping")
                    continue

                cell_data = (cell_data - mean) / std
                cell_dict[f"cell_{cell+1}"] = cell_data.T
                

            animal_clean_dict_activity[f"animal_{animal_idx+1}"] = cell_dict
            animal_vel_dict[f"animal_{animal_idx+1}"] = {"Velocity":vel_clean}
            animal_lick_dict[f"animal_{animal_idx+1}"] = {"Licks":lick_clean}

        return animal_clean_dict_activity, animal_vel_dict, animal_trials_original, animal_trials_clean, trials_to_remove_local, animal_lick_dict



In [24]:
for animal in animal_vel_dict:
    for key in animal_vel_dict[animal]:
        data_array = np.array(animal_vel_dict[animal][key])
        print(np.any(np.isnan(data_array)))

False
False
False
False
False
False
False
False
False
False


In [25]:
animal_clean_dict_activity

for animal in animal_clean_dict_activity:
    for cell in animal_clean_dict_activity[animal]:
        data_array = np.array(animal_clean_dict_activity[animal][cell])
        print(np.any(np.isnan(data_array)))

True
False
False
False
False
False
True
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
True
False
False
False
False
False
False
True
False
False
False
False
True
False
False
False
False
False
False
False
False
False
False
False
False
False
True
False
False
False
False
False
False
False
False
False
False
False
False
False
True
False
False
False
True
False
False
False
True
False
False
False
True
False
False
False
False
False
False
False
False
False
False


In [ ]:
filepath = '/Users/michaelfinch/CA1-interneuron-GLM/datasets/SSTindivsomata_GLM.mat'

animal_clean_dict_activity, animal_vel_dict, animal_trials_original, animal_trials_clean, trials_to_remove_local, animal_lick_dict = get_animal_clean_dict_activity(filepath, use_final=False)

GLM_params, predicted_activity_dict = fit_GLM_population(animal_vel_dict, animal_clean_dict_activity, quintile=None, regression='ridge', alphas=None)

residual_activity_dict_SST = get_residual_activity_dict(animal_clean_dict_activity, predicted_activity_dict)

animal_group.keys() <KeysViewHDF5 ['ShiftLrate', 'ShiftR', 'ShiftRunning', 'ShiftV']>
 -> zero or bad std for this cell, skipping
 -> zero or bad std for this cell, skipping
 -> zero or bad std for this cell, skipping
 -> zero or bad std for this cell, skipping
 -> zero or bad std for this cell, skipping
 -> zero or bad std for this cell, skipping
 -> zero or bad std for this cell, skipping
 -> zero or bad std for this cell, skipping
 -> zero or bad std for this cell, skipping
 -> zero or bad std for this cell, skipping


import numpy as np
import matplotlib.pyplot as plt
import os
import torch
import slicetca

# import utils as ut
# import plot as pt
plt.rcParams.update({'font.size': 12,
                     'axes.spines.right': False,
                     'axes.spines.top':   False,
                     'legend.frameon':    False,})

%load_ext autoreload
%autoreload 2


import sys
from scipy.stats import sem
sys.path.append('/Users/michaelfinch/CA1-interneuron-GLM')

from utils_TCA_clustering_scratchpad import *
from GLM_regression_plotting import *


from modelling_to_date_utils import *
from SliceTCA_example import *


GLM_params_SST, activity_dict_SST, double_predicted_activity_dict_SST, factors_dict_SST, filtered_factors_dict_SST, residual_activity_dict_SST = load_data_regular(file_path='/Users/michaelfinch/CA1-interneuron-GLM', name="SSTindivsomata_GLM", new_NDNF=False)
# GLM_params_EC, activity_dict_EC, double_predicted_activity_dict_EC, factors_dict_EC, filtered_factors_dict_EC, residual_activity_dict_EC = load_data_regular(file_path='/Users/michaelfinch/CA1-interneuron-GLM', name="EC_GLM", new_NDNF=False)

In [2]:
tensor_per_animal_list = []

for animal in residual_activity_dict_SST:
    cell_list = []
    for cell in residual_activity_dict_SST[animal]:
        cell_list.append(residual_activity_dict_SST[animal][cell])

    cells_array = np.array(cell_list)
    cells_array = cells_array.transpose(2, 0, 1)
    tensor_per_animal_list.append(cells_array)

In [3]:
per_num_latents_dict = {}

for i in range(1, 61):

    components = (0,i,0)

    model_per_animal_list = []

    for animal in range(len(tensor_per_animal_list)):

        cells_array = tensor_per_animal_list[animal]

        example_animal_tensor = torch.from_numpy(cells_array)

        components20, model_20 = slicetca.decompose(example_animal_tensor,
                                                    number_components=components, # (trials, neurons, time bins)
                                                    positive=False,learning_rate=1*10**-2, min_std=10**-5, max_iter=4000, iter_std=1000,seed=0)
        
        model_per_animal_list.append(model_20)

    per_num_latents_dict[i] = model_per_animal_list

Loss: 0.3501451653295509 : 100%|██████████| 4000/4000 [01:42<00:00, 38.99it/s] 
The model converged. Loss: 1.552049131870303e-06 :  36%|███▌      | 1449/4000 [00:24<00:43, 58.23it/s]
The model converged. Loss: 2.874954498249548e-05 :  41%|████      | 1623/4000 [00:31<00:45, 52.17it/s]
Loss: 0.16913223475434316 : 100%|██████████| 4000/4000 [01:19<00:00, 50.16it/s]
The model converged. Loss: 1.1900014414759877e-06 :  39%|███▊      | 1541/4000 [00:31<00:50, 49.04it/s]
Loss: 0.32495981770689053 : 100%|██████████| 4000/4000 [01:59<00:00, 33.53it/s]
The model converged. Loss: 2.6499351242536107e-06 :  36%|███▋      | 1457/4000 [00:46<01:21, 31.37it/s]
Loss: 0.07493636958875174 : 100%|██████████| 4000/4000 [01:32<00:00, 43.09it/s]
The model converged. Loss: 8.951974300406358e-06 :  37%|███▋      | 1465/4000 [00:45<01:18, 32.15it/s]
Loss: 0.253162000750076 : 100%|██████████| 4000/4000 [02:19<00:00, 28.75it/s]  
The model converged. Loss: 2.458561670348828e-05 :  35%|███▌      | 1414/4000 [00:5

KeyboardInterrupt: 